# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
import os

FEATURES = ["P_dyn", "VBs", "epsilon", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "comparison_template_deriv_against_baselines_only_definitive"
RAW_EQ = "g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093)"

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

import utils

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [ ]:
ace_columns = ['Bmag', 'Bx', 'By', 'Bz','Vp', 'Np', 'T']

dst_kyoto = utils.read_iaga_file('./data/dst-kyoto.txt', columns = ["DATE", "TIME", "DOY", "DST"])
dst_kyoto = dst_kyoto[~dst_kyoto.index.duplicated(keep='first')]

ace_imf = utils.read_data('./data/all_timeline', pattern_to_read=['csv', 'ace_imf_1h_'], print_info = True, return_separated = False)
ace_imf.columns = ['Bmag', 'Bx', 'By', 'Bz']
ace_imf = ace_imf[~ace_imf.index.duplicated(keep='first')]

ace_swepam = utils.read_data('./data/all_timeline', pattern_to_read=['csv', 'ace_swepam_1h_'], print_info = True, return_separated = False)                          
ace_swepam.columns = ['Vx', 'Vy', 'Vz', 'Vp', 'Np', 'T']
ace_swepam = ace_swepam.loc[:, ('Vp', 'Np', 'T')]
ace_swepam = ace_swepam[~ace_swepam.index.duplicated(keep='first')]
    
ace_data = ace_imf.join(ace_swepam)

all_data = ace_data.join(dst_kyoto['DST'])

all_data = all_data.interpolate()

data = compute_features(all_data)

Reading from file ./data/all_timeline/ace_imf_1h_1998.csv
Reading from file ./data/all_timeline/ace_imf_1h_1999.csv
Reading from file ./data/all_timeline/ace_imf_1h_2000.csv
Reading from file ./data/all_timeline/ace_imf_1h_2001.csv
Reading from file ./data/all_timeline/ace_imf_1h_2002.csv
Reading from file ./data/all_timeline/ace_imf_1h_2003.csv
Reading from file ./data/all_timeline/ace_imf_1h_2004.csv
Reading from file ./data/all_timeline/ace_imf_1h_2005.csv
Reading from file ./data/all_timeline/ace_imf_1h_2006.csv
Reading from file ./data/all_timeline/ace_imf_1h_2007.csv
Reading from file ./data/all_timeline/ace_imf_1h_2008.csv
Reading from file ./data/all_timeline/ace_imf_1h_2009.csv
Reading from file ./data/all_timeline/ace_imf_1h_2010.csv
Reading from file ./data/all_timeline/ace_imf_1h_2011.csv
Reading from file ./data/all_timeline/ace_imf_1h_2012.csv
Reading from file ./data/all_timeline/ace_imf_1h_2013.csv
Reading from file ./data/all_timeline/ace_imf_1h_2014.csv
Reading from f

In [4]:
def predict_and_plot_storm(model, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values

    res_eq = simulate_storm(model, storm_df)
    res_burton = baseline_models.burton_prediction(storm_df)
    res_obm = baseline_models.obm_prediction(storm_df)
    ddm1 = baseline_models.ddm1_prediction(storm_df)
    ddm2 = baseline_models.ddm2_prediction(storm_df)
    ddm3 = baseline_models.ddm3_prediction(storm_df)

    res_eq = res_eq[start:end]["DST_pred"].values
    res_burton = res_burton[start:end]["DST_pred"].values
    res_obm = res_obm[start:end]["DST_pred"].values
    res_ddm1 = ddm1[start:end]["DST_pred"].values
    res_ddm2 = ddm2[start:end]["DST_pred"].values
    res_ddm3 = ddm3[start:end]["DST_pred"].values

    # 2. Calculate Metrics
    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)
    fig.suptitle(
        rf"Evaluation for Equation: ${model.latex_str()}$", fontsize=18, wrap=True
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_eq,
        color="blue",
        linestyle="--",
        label="Equation",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_burton,
        color="yellow",
        linestyle="--",
        label="Burton",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_obm,
        color="green",
        linestyle="--",
        label="OBM",
        linewidth=1.5,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm1,
        color="orange",
        linestyle="--",
        label="DDM1",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm2,
        color="purple",
        linestyle="--",
        label="DDM2",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm3,
        color="cyan",
        linestyle="--",
        label="DDM3",
        linewidth=1.5,
    )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction", fontsize=18)
    axs[0].tick_params(axis='both', which='major', labelsize=14)
    axs[0].tick_params(axis='both', which='minor', labelsize=10)
    axs[0].legend(fontsize = 16)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_xlabel("Date", fontsize=16)
    axs[0].set_ylabel("Dst (nT)", fontsize=16)
    axs[0].xaxis.set_major_locator(MultipleLocator(2))

    # Column 2: Prediction Error
    diff_eq = res_eq - y_true
    diff_burton = res_burton - y_true
    diff_obm = res_obm - y_true
    diff_ddm1 = res_ddm1 - y_true
    diff_ddm2 = res_ddm2 - y_true
    diff_ddm3 = res_ddm3 - y_true


    axs[1].plot(storm_df[start:end].index, diff_eq, color="blue", label="Eq Error")
    axs[1].plot(
        storm_df[start:end].index, diff_burton, color="yellow", label="Burton Error"
    )
    axs[1].plot(storm_df[start:end].index, diff_obm, color="green", label="OBM Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm1, color="orange", label="DDM1 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm2, color="purple", label="DDM2 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm3, color="cyan", label="DDM3 Error")
    axs[1].axhline(0, color="black", linestyle="--")
    axs[1].set_title("Prediction Error", fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].set_xlabel("Date", fontsize=16)

    title_metrics = (
        f"Error Comparison\n"
        f"Eq: RMSE {m_eq[0]:.2f} | MAE {m_eq[1]:.2f} | R2 {m_eq[2]:.2f} | CC {m_eq[3]:.2f} | BFE {m_eq[4]:.2f}\n"
        f"Burton: RMSE {m_burton[0]:.2f} | MAE {m_burton[1]:.2f} | R2 {m_burton[2]:.2f} | CC {m_burton[3]:.2f} | BFE {m_burton[4]:.2f}\n"
        f"OBM: RMSE {m_obm[0]:.2f} | MAE {m_obm[1]:.2f} | R2 {m_obm[2]:.2f} | CC {m_obm[3]:.2f} | BFE {m_obm[4]:.2f}\n"
        f"DDM1: RMSE {m_ddm1[0]:.2f} | MAE {m_ddm1[1]:.2f} | R2 {m_ddm1[2]:.2f} | CC {m_ddm1[3]:.2f} | BFE {m_ddm1[4]:.2f}\n"
        f"DDM2: RMSE {m_ddm2[0]:.2f} | MAE {m_ddm2[1]:.2f} | R2 {m_ddm2[2]:.2f} | CC {m_ddm2[3]:.2f} | BFE {m_ddm2[4]:.2f}\n"
        f"DDM3: RMSE {m_ddm3[0]:.2f} | MAE {m_ddm3[1]:.2f} | R2 {m_ddm3[2]:.2f} | CC {m_ddm3[3]:.2f} | BFE {m_ddm3[4]:.2f}"
    )
    axs[1].set_title(title_metrics, fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].set_xlabel("Date", fontsize=16)
    axs[1].legend(fontsize=16)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis='both', which='major', labelsize=14)
    axs[1].tick_params(axis='both', which='minor', labelsize=10)
    
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        [res_eq, res_burton, res_obm, res_ddm1, res_ddm2, res_ddm3],
        ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"],
        ["blue", "yellow", "green", "orange", "purple", "cyan"],
        fontsize = 16   
    )

    plt.savefig(save_path)
    plt.close()


In [5]:
def save_prediction_data(model, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eq = simulate_storm(model, storm_df)
    if model.is_template:
        pred_dst_eq = pred_dst_eq[start:end][
            ["DST_pred", "dDST", "injection_component", "decay_component"]
        ]
    else:
        pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
    # 3. Baseline Predictions (Burton & OBM)
    pred_dst_burton = baseline_models.burton_prediction(storm_df)
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred", "dDST"]]
    pred_dst_burton.columns = ["DST_pred_burton", "dDST_burton"]
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred_burton", "dDST_burton"]]
    pred_dst_obm = baseline_models.obm_prediction(storm_df)
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred", "dDST"]]
    pred_dst_obm.columns = ["DST_pred_obm", "dDST_obm"]
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred_obm", "dDST_obm"]]
    pred_dst_ddm1 = baseline_models.ddm1_prediction(storm_df)    
    pred_dst_ddm1.columns = ["DST_pred_ddm1", "dDST_ddm1"]
    pred_dst_ddm1 = pred_dst_ddm1[start:end][["DST_pred_ddm1", "dDST_ddm1"]]
    pred_dst_ddm2 = baseline_models.ddm2_prediction(storm_df)
    pred_dst_ddm2.columns = ["DST_pred_ddm2", "dDST_ddm2"]
    pred_dst_ddm2 = pred_dst_ddm2[start:end][["DST_pred_ddm2", "dDST_ddm2"]]
    pred_dst_ddm3 = baseline_models.ddm3_prediction(storm_df)
    pred_dst_ddm3.columns = ["DST_pred_ddm3", "dDST_ddm3"]
    pred_dst_ddm3 = pred_dst_ddm3[start:end][["DST_pred_ddm3", "dDST_ddm3"]]


    # 4. Construct Comprehensive DataFrame

    if model.is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,
            }
        ).set_index("Timestamp")
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,

            }
        ).set_index("Timestamp")

    results_df.to_csv(output_path)
    return results_df

## Test storms

In [6]:
storms = []

model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION_DEFINITIVE
storm_indices = []
for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'w') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(model.latex_str())}\n')

100%|██████████| 1/1 [00:01<00:00,  1.62s/it]


In [7]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,55.0,17.429458,13.783357,0.748936,0.886955,19.558619,22.567406,19.621359,0.579099,0.890685,...,20.994874,17.672795,0.635713,0.870809,22.088199,22.435193,18.173523,0.584016,0.864248,25.379149
1,Mean,17.429458,13.783357,0.748936,0.886955,19.558619,22.567406,19.621359,0.579099,0.890685,...,20.994874,17.672795,0.635713,0.870809,22.088199,22.435193,18.173523,0.584016,0.864248,25.379149
2,Global,17.429458,13.783357,0.748936,0.886955,19.558619,22.567406,19.621359,0.579099,0.890685,...,20.994874,17.672795,0.635713,0.870809,22.088199,22.435193,18.173523,0.584016,0.864248,25.379149


In [8]:
display(summary_df.iloc[-2:, :].style.format({col: "{:.2f}" for col in columns}))

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,Burton_BFE,OBM_RMSE,OBM_MAE,OBM_R2,OBM_CC,OBM_BFE,DDM1_RMSE,DDM1_MAE,DDM1_R2,DDM1_CC,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
1,Mean,17.43,13.78,0.75,0.89,19.56,22.57,19.62,0.58,0.89,19.08,17.19,14.44,0.76,0.90,16.81,21.69,17.79,0.61,0.87,23.43,20.99,17.67,0.64,0.87,22.09,22.44,18.17,0.58,0.86,25.38
2,Global,17.43,13.78,0.75,0.89,19.56,22.57,19.62,0.58,0.89,19.08,17.19,14.44,0.76,0.90,16.81,21.69,17.79,0.61,0.87,23.43,20.99,17.67,0.64,0.87,22.09,22.44,18.17,0.58,0.86,25.38


In [9]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,55.0,19.558619,19.077484,16.806135,23.43336,22.088199,25.379149
1,Mean,19.558619,19.077484,16.806135,23.43336,22.088199,25.379149
2,Global,19.558619,19.077484,16.806135,23.43336,22.088199,25.379149


In [10]:
print(summary_df.loc[summary_df["Storm Index"].isin([55, 57, 68, 'Mean'])].to_latex(index=False, float_format="%.3f"))

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
Storm Index & Equation_RMSE & Equation_MAE & Equation_R2 & Equation_CC & Equation_BFE & Burton_RMSE & Burton_MAE & Burton_R2 & Burton_CC & Burton_BFE & OBM_RMSE & OBM_MAE & OBM_R2 & OBM_CC & OBM_BFE & DDM1_RMSE & DDM1_MAE & DDM1_R2 & DDM1_CC & DDM1_BFE & DDM2_RMSE & DDM2_MAE & DDM2_R2 & DDM2_CC & DDM2_BFE & DDM3_RMSE & DDM3_MAE & DDM3_R2 & DDM3_CC & DDM3_BFE \\
\midrule
55.000 & 17.429 & 13.783 & 0.749 & 0.887 & 19.559 & 22.567 & 19.621 & 0.579 & 0.891 & 19.077 & 17.193 & 14.438 & 0.756 & 0.897 & 16.806 & 21.695 & 17.791 & 0.611 & 0.875 & 23.433 & 20.995 & 17.673 & 0.636 & 0.871 & 22.088 & 22.435 & 18.174 & 0.584 & 0.864 & 25.379 \\
Mean & 17.429 & 13.783 & 0.749 & 0.887 & 19.559 & 22.567 & 19.621 & 0.579 & 0.891 & 19.077 & 17.193 & 14.438 & 0.756 & 0.897 & 16.806 & 21.695 & 17.791 & 0.611 & 0.875 & 23.433 & 20.995 & 17.673 & 0.636 & 0.871 & 22.088 & 22.435 & 18.174 & 0.584 & 0.864 & 25.379 \\
\bottomrule
\end{tabular}

